# Advanced Pitch Statistics from Statcast

This notebook explores the advanced pitch-level data used to build pitcher and batter profiles.

**Pitcher Arsenal Stats:**
- Velocity, spin, movement by pitch type
- Pitch usage percentages
- Whiff%, CSW%, zone%, chase induced

**Batter Profile Stats:**
- Performance vs pitch types (fastball, breaking, offspeed)
- Performance vs velocity buckets (90-93, 94-96, 97+)
- Performance vs movement profiles (high rise, heavy sweep, heavy drop)
- Contact quality (exit velo, xwOBA)

In [ ]:
import pandas as pd
import numpy as np

from mlb_data import (
    get_statcast_pitches,
    get_pitcher_arsenal,
    get_pitcher_pitch_type_stats,
    get_batter_pitch_stats,
    get_batter_pitch_type_stats,
    get_plate_appearances,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Date range for data collection
# Start with a smaller range for exploration (1-2 months)
START_DATE = '2024-05-01'
END_DATE = '2024-06-30'

## 1. Raw Statcast Data

First, let's fetch the raw pitch-level data and explore what's available.

In [ ]:
# Fetch raw Statcast pitches
# This may take a few minutes for large date ranges
pitches = get_statcast_pitches(START_DATE, END_DATE)

print(f"Total pitches: {len(pitches):,}")
print(f"Date range: {pitches['game_date'].min()} to {pitches['game_date'].max()}")
print(f"\nColumns ({len(pitches.columns)}):")
print(pitches.columns.tolist())

In [ ]:
# Key pitch characteristic columns
pitch_cols = [
    'pitch_type', 'release_speed', 'release_spin_rate',
    'pfx_x', 'pfx_z', 'release_extension',
    'plate_x', 'plate_z', 'zone',
    'description', 'events', 'type',
]

pitches[pitch_cols].head(20)

In [ ]:
# Pitch type distribution
print("Pitch type distribution:")
pitch_counts = pitches['pitch_type'].value_counts()
pitch_pcts = pitches['pitch_type'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': pitch_counts,
    'pct': pitch_pcts.round(1)
}).head(15)

In [ ]:
# Pitch outcome descriptions
print("Pitch descriptions (outcomes):")
pitches['description'].value_counts()

## 2. Pitcher Arsenal Profiles

Aggregate pitch-level data into pitcher profiles with:
- Fastball characteristics (velocity, spin, movement)
- Breaking ball characteristics
- Offspeed characteristics
- Overall effectiveness metrics (whiff%, CSW%, etc.)

In [ ]:
# Get pitcher arsenal stats (uses cached pitches)
pitcher_arsenal = get_pitcher_arsenal(
    START_DATE, END_DATE,
    min_pitches=200,
    pitches_df=pitches
)

print(f"Pitchers with arsenal data: {len(pitcher_arsenal)}")
pitcher_arsenal.head(10)

In [ ]:
# All available arsenal columns
print("Pitcher arsenal columns:")
print(pitcher_arsenal.columns.tolist())

In [ ]:
# Summary statistics for key arsenal metrics
arsenal_stats = [
    'fb_velo_avg', 'fb_velo_max', 'fb_spin_avg', 'fb_vmov_avg',
    'brk_velo_avg', 'brk_hmov_avg', 'brk_vmov_avg',
    'off_velo_avg', 'off_velo_diff',
    'fb_usage_pct', 'brk_usage_pct', 'off_usage_pct',
    'whiff_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced',
]

available_stats = [c for c in arsenal_stats if c in pitcher_arsenal.columns]
pitcher_arsenal[available_stats].describe().round(2)

In [ ]:
# Top 10 hardest throwers
print("Top 10 by fastball velocity:")
pitcher_arsenal.nlargest(10, 'fb_velo_avg')[[
    'pitcher_name', 'p_throws', 'total_pitches',
    'fb_velo_avg', 'fb_velo_max', 'fb_spin_avg', 'whiff_pct'
]]

In [ ]:
# Top 10 by whiff rate
print("Top 10 by whiff percentage:")
pitcher_arsenal.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'p_throws', 'total_pitches',
    'fb_velo_avg', 'brk_usage_pct', 'whiff_pct', 'csw_pct'
]]

## 3. Pitcher Stats by Pitch Type

Detailed breakdown for each pitch type a pitcher throws.

In [ ]:
# Get per-pitch-type stats
pitcher_pitch_types = get_pitcher_pitch_type_stats(
    START_DATE, END_DATE,
    min_pitches=50,
    pitches_df=pitches
)

print(f"Pitcher-pitch type combinations: {len(pitcher_pitch_types)}")
pitcher_pitch_types.head(15)

In [ ]:
# Best sliders by whiff rate
sliders = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'SL']
print("Top 10 sliders by whiff rate:")
sliders.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'hmov_avg', 'vmov_avg', 'whiff_pct'
]]

In [ ]:
# Best changeups by whiff rate
changeups = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'CH']
print("Top 10 changeups by whiff rate:")
changeups.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'vmov_avg', 'whiff_pct'
]]

## 4. Batter Pitch Performance Profiles

How batters perform against different pitch characteristics:
- By pitch type group (fastball, breaking, offspeed)
- By velocity bucket (90-93, 94-96, 97+)
- By movement type (high rise, heavy sweep, heavy drop)
- By pitcher handedness (vs LHP, vs RHP)

In [ ]:
# Get batter pitch profiles
batter_profiles = get_batter_pitch_stats(
    START_DATE, END_DATE,
    min_pitches=200,
    pitches_df=pitches
)

print(f"Batters with pitch profiles: {len(batter_profiles)}")
batter_profiles.head(10)

In [ ]:
# All available batter profile columns
print("Batter profile columns:")
print(batter_profiles.columns.tolist())

In [ ]:
# Summary of key batter metrics
batter_stats = [
    'whiff_pct', 'contact_pct', 'chase_pct', 'zone_swing_pct',
    'fastball_whiff_pct', 'breaking_whiff_pct', 'offspeed_whiff_pct',
    'velo_97_plus_whiff_pct', 'high_rise_whiff_pct', 'heavy_sweep_whiff_pct',
    'avg_exit_velo', 'xwoba',
]

available_stats = [c for c in batter_stats if c in batter_profiles.columns]
batter_profiles[available_stats].describe().round(3)

In [ ]:
# Best contact rates (lowest whiff)
print("Top 10 contact rates (lowest whiff%):")
batter_profiles.nsmallest(10, 'whiff_pct')[[
    'batter_id', 'stand', 'total_pitches',
    'whiff_pct', 'contact_pct', 'chase_pct', 'avg_exit_velo'
]]

In [ ]:
# Batters who struggle vs 97+ velocity
if 'velo_97_plus_whiff_pct' in batter_profiles.columns:
    has_velo_data = batter_profiles['velo_97_plus_whiff_pct'].notna()
    print("Batters who struggle most vs 97+ mph:")
    batter_profiles[has_velo_data].nlargest(10, 'velo_97_plus_whiff_pct')[[
        'batter_id', 'stand', 'velo_97_plus_pitches',
        'velo_97_plus_whiff_pct', 'whiff_pct'
    ]]

In [ ]:
# Batters who struggle vs breaking balls
if 'breaking_whiff_pct' in batter_profiles.columns:
    has_brk_data = batter_profiles['breaking_whiff_pct'].notna()
    print("Batters who struggle most vs breaking balls:")
    batter_profiles[has_brk_data].nlargest(10, 'breaking_whiff_pct')[[
        'batter_id', 'stand', 'breaking_pitches',
        'breaking_whiff_pct', 'whiff_pct'
    ]]

## 5. Batter Stats by Pitch Type

Detailed breakdown for each pitch type a batter faces.

In [ ]:
# Get per-pitch-type stats for batters
batter_pitch_types = get_batter_pitch_type_stats(
    START_DATE, END_DATE,
    min_pitches=30,
    pitches_df=pitches
)

print(f"Batter-pitch type combinations: {len(batter_pitch_types)}")
batter_pitch_types.head(15)

In [ ]:
# Best fastball hitters (lowest whiff on FF)
ff_stats = batter_pitch_types[batter_pitch_types['pitch_type'] == 'FF']
if len(ff_stats) > 0 and 'xwoba' in ff_stats.columns:
    ff_qualified = ff_stats[ff_stats['pitch_count'] >= 100]
    print("Best fastball hitters by xwOBA:")
    ff_qualified.nlargest(10, 'xwoba')[[
        'batter_id', 'pitch_count', 'whiff_pct', 'contact_pct',
        'avg_exit_velo', 'xwoba'
    ]]

## 6. Plate Appearance Outcomes

Extract completed plate appearances with outcomes (K, BB, 1B, 2B, 3B, HR, OUT).

In [ ]:
# Get plate appearances
plate_apps = get_plate_appearances(
    START_DATE, END_DATE,
    pitches_df=pitches
)

print(f"Total plate appearances: {len(plate_apps):,}")
plate_apps.head(15)

In [ ]:
# Outcome distribution
print("PA outcome distribution:")
outcome_counts = plate_apps['outcome'].value_counts()
outcome_pcts = plate_apps['outcome'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': outcome_counts,
    'pct': outcome_pcts.round(1)
})

In [ ]:
# Unique matchup counts
print(f"Unique pitchers: {plate_apps['pitcher_id'].nunique():,}")
print(f"Unique batters: {plate_apps['batter_id'].nunique():,}")

# Pitcher-batter combinations
matchups = plate_apps.groupby(['pitcher_id', 'batter_id']).size()
print(f"Unique pitcher-batter matchups: {len(matchups):,}")
print(f"\nPAs per matchup distribution:")
matchups.describe()

## 7. Save Data

Save the computed profiles for use in model training.

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path('../data/statcast')
output_dir.mkdir(parents=True, exist_ok=True)

# Save profiles
pitcher_arsenal.to_csv(output_dir / 'pitcher_arsenal.csv', index=False)
pitcher_pitch_types.to_csv(output_dir / 'pitcher_pitch_types.csv', index=False)
batter_profiles.to_csv(output_dir / 'batter_profiles.csv', index=False)
batter_pitch_types.to_csv(output_dir / 'batter_pitch_types.csv', index=False)
plate_apps.to_csv(output_dir / 'plate_appearances.csv', index=False)

print(f"Saved data to {output_dir.resolve()}:")
print(f"  - pitcher_arsenal.csv: {len(pitcher_arsenal)} rows")
print(f"  - pitcher_pitch_types.csv: {len(pitcher_pitch_types)} rows")
print(f"  - batter_profiles.csv: {len(batter_profiles)} rows")
print(f"  - batter_pitch_types.csv: {len(batter_pitch_types)} rows")
print(f"  - plate_appearances.csv: {len(plate_apps)} rows")